# Spike: Locating Prep for AI Settings in `getDefinition`

Read-only inspection of a Fabric semantic model's `Copilot/` folder via the REST `getDefinition` endpoint.

**Corrected hypothesis (after verification against Microsoft Learn):**

AI Instructions, AI Data Schema, and Verified Answers do not live in TMDL annotations. They live in a `Copilot/` folder that is sibling to `definition/` inside the semantic model. The Fabric REST API `getDefinition` endpoint returns the `Copilot/` parts in the same payload envelope as TMDL parts, so a single API call surfaces everything.

**What this notebook does**

1. Authenticates against Fabric.
2. Calls `POST .../semanticModels/{id}/getDefinition` for a target model.
3. Lists every part path in the response.
4. Filters for `Copilot/` parts and labels each by primitive (instructions, schema, verified answers, etc.).
5. Prints a snippet of each Copilot part so you can confirm content shape (Markdown for instructions, JSON for the rest).

**What this notebook does not do**

It does not call `updateDefinition`. Writes are out of scope; the future `CopilotWriter` covers the round-trip path.

**Prerequisites**

- Run inside a Fabric notebook (or anywhere with a Power BI / Fabric bearer token).
- The target semantic model has Q&A enabled and at least one AI Instruction and one Verified Answer configured. Without configured settings, absence of a hit proves nothing.
- `getDefinition` requires Member / Contributor / Admin on the workspace, or Build permission on the model.

## 1. Install

In [ ]:
%pip install fabric-ai-meta

## 2. Configure target model

Replace the placeholders below with your workspace and semantic model identifiers (GUIDs).

In [ ]:
WORKSPACE_ID = "YOUR_WORKSPACE_GUID"
MODEL_ID = "YOUR_SEMANTIC_MODEL_GUID"

## 3. Acquire a Fabric bearer token

Inside a Fabric notebook, `notebookutils.credentials.getToken('pbi')` returns a token scoped for Power BI / Fabric. Outside Fabric, use an `azure.identity` credential (e.g., `InteractiveBrowserCredential` or a service principal) acquired with the `https://api.fabric.microsoft.com/.default` scope.

In [ ]:
try:
    import notebookutils  # type: ignore[import-not-found]
    token = notebookutils.credentials.getToken("pbi")
    credential = token  # TMDLClient accepts a raw bearer string.
    print("Acquired Fabric notebook token.")
except ImportError:
    from azure.identity import InteractiveBrowserCredential
    credential = InteractiveBrowserCredential()
    print("Falling back to InteractiveBrowserCredential.")

## 4. Fetch the model definition and list parts

Call `getDefinition` and inspect the file list. Expect to see a mix of `definition/...tmdl` files and `Copilot/...` files (when Q&A and Prep for AI are configured).

In [ ]:
from fabric_ai_meta.writeback.tmdl_client import TMDLClient

client = TMDLClient(credential, WORKSPACE_ID)
definition = client.get_definition(MODEL_ID)

paths = [p["path"] for p in definition["definition"]["parts"]]
tmdl_paths = [p for p in paths if p.startswith("definition/")]
copilot_paths = [p for p in paths if p.startswith("Copilot/")]
other_paths = [p for p in paths if p not in tmdl_paths and p not in copilot_paths]

print(f"Total parts: {len(paths)}")
print(f"  TMDL parts:    {len(tmdl_paths)}")
print(f"  Copilot parts: {len(copilot_paths)}")
print(f"  Other parts:   {len(other_paths)}")
if copilot_paths:
    print("\nCopilot parts:")
    for p in copilot_paths:
        print(f"  - {p}")

## 5. Identify Prep for AI primitives

`find_prep_for_ai_settings` filters for `Copilot/` parts and labels each by primitive (`ai_instructions`, `ai_data_schema`, `verified_answers`, `example_prompts`, `copilot_settings`, `copilot_version`).

In [ ]:
result = client.find_prep_for_ai_settings(definition)

if result is None:
    print("No Copilot parts present. Either Prep for AI has not been configured")
    print("on this model, or Q&A is not enabled (definition.pbism qnaEnabled flag).")
else:
    print(f"Found {len(result['matches'])} Copilot parts:\n")
    for match in result["matches"]:
        print(f"--- {match['primitive']:20s}  {match['path']}")
        print(match["snippet"])
        print()

## 6. Inspect AI Instructions in full

AI Instructions are Markdown. Print the full file contents so you can see how Microsoft formats them in the live model.

In [ ]:
import base64

for part in definition["definition"]["parts"]:
    if part["path"] == "Copilot/Instructions/instructions.md":
        text = base64.b64decode(part["payload"]).decode("utf-8", errors="replace")
        print(text)
        break
else:
    print("No Copilot/Instructions/instructions.md found in this model.")

## 7. Record findings

Use the output of cells 4-6 to confirm the live model's `Copilot/` shape:

- Confirm the path layout matches the documented `Copilot/` tree.
- Note any additional files Microsoft serializes that are not yet in `COPILOT_PATH_PREFIXES`.
- Capture the AI Instructions Markdown shape (headings, bullet structure, conventions) for the future `CopilotWriter` to emit compatibly.
- Record whether the model is Import / DirectQuery / Direct Lake; the refresh-latency caveat (DQ + Direct Lake = once per day before Copilot picks up changes) only matters for non-Import models.

Once the path layout and Markdown shape are confirmed, the next step is a `CopilotWriter` that reads `Copilot/Instructions/instructions.md` (and the JSON peers), edits in memory, and writes back through `updateDefinition` while round-tripping every other part byte-for-byte.